# Heatmap 3

Heatmap and clustermap for each of 1-5. Run the preamble, then any numbered section.

In [ ]:
import os
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import NLProcessing
from collections import Counter
from datetime import datetime
from matplotlib import font_manager
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

date = datetime.today().strftime('%Y%m%d')

In [ ]:
filedir = "\\Data Compilation\\Climbing_New\\"
laptop = "C:\\Users\\lnico\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee"
homecomp = "D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee"   # TODO: update when home Dropbox moves to NUS Dropbox
labcomp = "C:\\Users\\User\\NUS Dropbox\\acclab\\Nicole M Lee"
specifiedpath = labcomp

openPath = specifiedpath + filedir
newfile2 = openPath + "Compilation with delta\\2025deltagcollection\\"

for f in font_manager.findSystemFonts(fontpaths=["fonts"]):
    font_manager.fontManager.addfont(f)
plt.rcParams["font.family"] = "Inter"
matplotlib.rcParams['svg.fonttype'] = 'none'

## Load

In [ ]:
files2 = os.listdir(newfile2)
totalfile = pd.DataFrame()

for n in files2:
    openfile = pd.read_csv(newfile2 + n)
    openfile['responder'] = n.split(" x ")[1].split("_")[0]
    totalfile = pd.concat([totalfile, openfile], axis=0).reset_index(drop=True)

totalfile['genotypeandresponder'] = totalfile["MBON"] + "_" + totalfile['responder']

onlymeans = totalfile.loc[:, ~totalfile.columns.str.endswith('_bootstrap')].drop_duplicates().reset_index(drop=True)
mbononly = onlymeans[~onlymeans['MBON'].isin(['R58', 'Th-Gal4', 'R76B09', 'VT999036'])]

cols_only = ['MBON', 'responder','genotypeandresponder', 'height_deltag', 'speed_deltag',
       'bspeed_deltag', 'maxvelocity_deltag',
       'straightindex_deltag', 'meanbout_deltag', 'bout_deltag']

RENAME = {'speed': 'Speed', 'bspeed': 'Bout speed', 'pausepos': 'Pause position', 'bout': '# Bouts', 'meanbout': "Mean bout time",
          'straightindex': 'Straightness Index', 'height': 'Avg height', 'maxvelocity': 'Max velocity'}

## Lobe location

In [ ]:
MBONList = list(set([yy.split(" ")[0] for yy in files2]))

csvfile = pd.read_csv(specifiedpath + "\\Data Compilation\\MBONlist.csv").astype('string')
for n, k in zip(["B", "y", "a"], ['\u03b2', "\u03b3", "\u03b1"]):
    csvfile['Lobe'] = csvfile['Lobe'].str.replace(n, k)

lobelocation = pd.DataFrame()
for m in MBONList:
    lobeloc = pd.DataFrame()
    lobeloc['MBON'] = [m]
    lobeloc['Lobe_location'] = [NLProcessing.find_number(csvfile, m, "Lobe")]
    lobeloc['MBON number'] = [NLProcessing.find_number(csvfile, m, "MBON number").strip()]
    lobeloc['Neurotransmitter'] = [NLProcessing.find_number(csvfile, m, "Neurotransmitter")]
    lobelocation = pd.concat([lobelocation, lobeloc])

lobelocation = lobelocation.reset_index(drop=True)


def addlobes(df):
    lobloclst = []
    mbonloclst = []
    out = df.copy()
    for n in df['MBON']:
        lobloclst.append(lobelocation[lobelocation['MBON'] == n]['Lobe_location'].values[0])
        mbonloclst.append(lobelocation[lobelocation['MBON'] == n]['MBON number'].values[0])
    out['Lobe'] = lobloclst
    out['Name'] = mbonloclst
    return out.sort_values(by = "Name", ascending=True).reset_index(drop=True)

## Functions

In [ ]:
def plot_heatmap(df_singleresponder_lobeloc, responderrename, colormap, responder, paired=False, vmin=-1.5, vmax=1.5):
    df60_other = df_singleresponder_lobeloc.set_index('Name').drop(['responder', 'MBON', 'Lobe', 'genotypeandresponder'], axis = 1).copy()

    fig1,  ax4 = plt.subplots(figsize=(10, max(4, len(df60_other) * 0.32)))

    newdf_singleresponder_lobeloc = pd.DataFrame()
    naming = df_singleresponder_lobeloc['MBON'].to_frame().rename(columns = {"MBON": ""})
    lobing = df_singleresponder_lobeloc['Lobe'].to_frame().rename(columns = {"Lobe": ""})
    newlabel =(naming.values + "\n " + lobing.values).tolist()
    newdf_singleresponder_lobeloc['Merge'] = newlabel

    xticklabeldf_other = df60_other.rename(columns=RENAME)
    xticklabellist_other = xticklabeldf_other.columns.tolist()

    cax2 = inset_axes(ax4,
                     width="15%",
                     height="0.8%",
                     loc='lower right',
                     bbox_to_anchor=(0.12, 1.05, 1, 1),
                     bbox_transform=ax4.transAxes,
                     borderpad=-2,
                     )

    sns.set_style("whitegrid", {'axes.grid' : False})

    j7_right = sns.heatmap(df60_other, ax = ax4, annot=True, fmt=".1f", vmin = vmin, vmax = vmax, cmap=colormap, linewidths=0.0, edgecolor = "none",  xticklabels=xticklabellist_other, yticklabels=True,
                     clip_on=False, cbar_ax=cax2, cbar_kws = dict(orientation = "horizontal", ticks = [vmin, 0, vmax] if vmin < 0 else [vmin, vmax]), annot_kws={"size": 14, })

    cbar = j7_right.collections[0].colorbar
    cbar.ax.tick_params(labelsize=8)
    for label in cbar.ax.get_xticklabels():
        label.set_family('Inter')

    j7_right.set_ylabel('')
    j7_right.set_yticklabels(j7_right.get_yticklabels(), va='center', rotation = 0, fontsize = 13)
    j7_right.set_xticklabels(j7_right.get_xticklabels(), fontsize = 11)
    NLProcessing.wrap_labels(j7_right, 5)
    j7_right.tick_params(axis='both', which='both', length=0)

    j7_right.axhline(y = 0, color = 'k', linewidth = 5)
    j7_right.axhline(y = len(df60_other), color = 'k', linewidth = 5)
    j7_right.axvline(x = 0, color = 'k', linewidth = 5)
    j7_right.axvline(x = len(df60_other.columns), color = 'k', linewidth = 5)

    j8_right = j7_right.twinx()
    j8_right.set_ylim([0,j7_right.get_ylim()[0]])
    j8_right.set_yticks(j7_right.get_yticks())

    labellinglist = newdf_singleresponder_lobeloc['Merge'].iloc[::-1]
    labelledlist = []
    for n in labellinglist:
        if paired:
            labelledlist.extend(("", n[0]))
        else:
            labelledlist.extend(n)
    j8_right.set_yticklabels(labelledlist, fontsize=9)
    j8_right.spines['top'].set_visible(False)
    j8_right.spines['right'].set_visible(False)
    j8_right.spines['bottom'].set_visible(False)
    j8_right.spines['left'].set_visible(False)

    fig1.suptitle('Plot of MBONs > ' + responderrename +' and their effects \n across locomotor reactivity parameters for Climbing Assay', x=0.5, y=1.0, weight='bold', fontsize =16)
    fig1.tight_layout()
    plt.savefig(openPath + "images\\" + date + "_" + responder + "_deltagheatmapwithlobelocations.svg", bbox_inches = "tight")
    plt.show()

In [ ]:
def parse_lobes(lobe_str):
    if pd.isna(lobe_str) or lobe_str == '':
        return ['Unknown']
    lobe_str_lower = lobe_str.lower()
    lobes = []
    if 'calyx' in lobe_str_lower: lobes.append('Calyx')
    if "\u03b1'" in lobe_str or "a'" in lobe_str_lower: lobes.append("\u03b1'")
    if "\u03b2'" in lobe_str or "b'" in lobe_str_lower: lobes.append("\u03b2'")
    if any(c in lobe_str_lower for c in ['a1', 'a2', 'a3', '\u03b11', '\u03b12', '\u03b13']) and "\u03b1'" not in lobes: lobes.append('\u03b1')
    if any(c in lobe_str_lower for c in ['b1', 'b2', 'b3', '\u03b21', '\u03b22', '\u03b23']) and "\u03b2'" not in lobes: lobes.append('\u03b2')
    if any(c in lobe_str_lower for c in ['y1', 'y2', 'y3', 'y4', 'y5', '\u03b31', '\u03b32', '\u03b33', '\u03b34', '\u03b35']): lobes.append('\u03b3')
    return lobes if lobes else ['Unknown']


def shorten_mbon_numbers(mbon_str):
    if pd.isna(mbon_str) or mbon_str == '':
        return mbon_str
    parts = [p.strip() for p in mbon_str.split(',')]
    if len(parts) <= 1:
        return mbon_str
    return ', '.join([parts[0]] + [p.replace('MBON', '').strip() for p in parts[1:]])


lobe_palette = {'\u03b1': '#E69F00', "\u03b1'": '#F0E442', '\u03b2': '#009E73', "\u03b2'": '#CC79A7', '\u03b3': '#999999', 'Calyx': '#000000', 'Unknown': '#FFFFFF'}
dendrogram_size = 0.08
gap = 0.008


def plot_clustermap(df_singleresponder_lobeloc, responderrename, colormap, responder, vmin=-1.5, vmax=1.5, methodlist=['complete'], metriclist=['euclidean']):
    df601 = df_singleresponder_lobeloc.copy()
    df601['totalnaming'] = df601['MBON'] + "; " + df601['Name'] + "; " + df601['Lobe'] + "; " + df601['responder']
    df601newset = df601.set_index('totalnaming').drop(['responder', 'genotypeandresponder', 'Lobe', 'Name', 'MBON'], axis = 1)

    df601['Lobes_parsed'] = df601['Lobe'].apply(parse_lobes)
    df601['Name_short'] = df601['Name'].apply(shorten_mbon_numbers)

    xticklabeldf = df601newset.rename(columns=RENAME)
    xticklabellist = xticklabeldf.columns.tolist()

    for methodd in methodlist:
        for metricc in metriclist:
            print (methodd + " " + metricc)
            j7 = sns.clustermap(df601newset, cmap=colormap, metric = metricc, method = methodd,
                                yticklabels=False, vmin = vmin, vmax=vmax, xticklabels=xticklabellist,
                                row_cluster=True, col_cluster=False, annot = False,
                                row_colors=['#FFFFFF'] * len(df601newset),
                                dendrogram_ratio=(dendrogram_size, dendrogram_size),
                                cbar_kws = dict(orientation = 'horizontal', ticks = [vmin, vmax]),
                                figsize=(5, max(4, len(df601newset) * 0.22)), cbar_pos=(0, 0.87, .02, .1))

            ax = j7.ax_heatmap
            ax.set_ylabel('')
            ax.set_xticklabels(ax.get_xticklabels(), rotation = 0, fontsize = 10)
            NLProcessing.wrap_labels(ax, 8)
            j7.fig.suptitle('Clustermap using ' + methodd + "_" + metricc + ' : \u0394g of MBONS >' + responderrename + "(Climbing)", weight='bold', fontsize =16, y =1.0)

            row_colors_pos = j7.ax_row_colors.get_position()
            heatmap_pos = j7.ax_heatmap.get_position()
            j7.ax_row_colors.set_position([row_colors_pos.x0 + gap, row_colors_pos.y0, row_colors_pos.width, row_colors_pos.height])
            j7.ax_heatmap.set_position([heatmap_pos.x0 + gap*2, heatmap_pos.y0, heatmap_pos.width - gap*2, heatmap_pos.height])

            ax_row_colors = j7.ax_row_colors
            row_order = j7.dendrogram_row.reordered_ind
            ax_row_colors.clear()
            ax_row_colors.set_xlim(0, 1)
            ax_row_colors.set_ylim(0, len(row_order))
            ax_row_colors.invert_yaxis()
            ax_row_colors.axis('off')

            for i, idx in enumerate(row_order):
                lobes = df601['Lobes_parsed'].iloc[idx]
                n_lobes = len(lobes) if lobes else 1
                for j_lobe, lobe in enumerate(lobes):
                    rect = plt.Rectangle((j_lobe / n_lobes, i), 1 / n_lobes, 1,
                                         facecolor=lobe_palette.get(lobe, '#FFFFFF'), edgecolor='white', linewidth=0.5)
                    ax_row_colors.add_patch(rect)

            for i, idx in enumerate(row_order):
                ax.text(1.005, i + 0.5, df601['Name_short'].iloc[idx], ha='left', va='center', fontsize=9, transform=ax.get_yaxis_transform())

            legend_elements = [plt.Rectangle((0,0),1,1, facecolor=c, edgecolor='white', label=l) for l, c in lobe_palette.items() if l != 'Unknown']
            j7.fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.45, 0.95),
                          ncol=len(legend_elements), fontsize=9, frameon=False)

            plt.savefig(openPath + "images\\" + date + "_" + responder + "CLIMBING_clustermap_" + methodd + "_" + metricc + ".svg", bbox_inches = "tight")
            plt.show()

## 1. ACR

In [ ]:
responder = "ACR"
responderrename = "CsChrimson" if responder == "Chrimson2" else "GtACR1"
colormap = 'coolwarm'

dfoneresponder = mbononly[cols_only]
df_specificmbon = dfoneresponder[dfoneresponder['responder']=="ACR"].sort_values(by = "bspeed_deltag", ascending=True).reset_index(drop=True)
df_specificmbon.columns = df_specificmbon.columns.str.replace('_deltag', '')

df_singleresponder_lobeloc = addlobes(df_specificmbon)
tag = "ACR"

In [ ]:
plot_heatmap(df_singleresponder_lobeloc, responderrename, colormap, tag)

In [ ]:
plot_clustermap(df_singleresponder_lobeloc, responderrename, colormap, tag)

## 2. Chrimson2

In [ ]:
responder = "Chrimson2"
responderrename = "CsChrimson" if responder == "Chrimson2" else "GtACR1"
colormap = 'coolwarm'

dfoneresponder = mbononly[cols_only]
df_specificmbon = dfoneresponder[dfoneresponder['responder']=="Chrimson2"].sort_values(by = "bspeed_deltag", ascending=True).reset_index(drop=True)
df_specificmbon.columns = df_specificmbon.columns.str.replace('_deltag', '')

df_singleresponder_lobeloc = addlobes(df_specificmbon)
tag = "Chrimson2"

In [ ]:
plot_heatmap(df_singleresponder_lobeloc, responderrename, colormap, tag)

In [ ]:
plot_clustermap(df_singleresponder_lobeloc, responderrename, colormap, tag)

## 3. ACR and Chrimson2

In [ ]:
responderrename = "GtACR1 and CsChrimson"
colormap = 'coolwarm'

counts = Counter(mbononly['MBON'].to_list())
matchingsets = [value for value, count in counts.items() if count > 1]
matchdf = mbononly[mbononly['MBON'].isin(matchingsets)][cols_only].sort_values(by = ["MBON", "responder"]).reset_index(drop=True)
matchdf.columns = matchdf.columns.str.replace('_deltag', '')

df_singleresponder_lobeloc = addlobes(matchdf)
tag = "ACRChrimson2"

In [ ]:
plot_heatmap(df_singleresponder_lobeloc, responderrename, colormap, tag, paired=True)

In [ ]:
plot_clustermap(df_singleresponder_lobeloc, responderrename, colormap, tag)

## 4. Specific MBON list

In [ ]:
responder = "Chrimson2"
specificmbons = ["MBON01", "MBON11", "MBON14"]
responderrename = "CsChrimson" if responder == "Chrimson2" else "GtACR1"
colormap = 'coolwarm'

dfoneresponder = mbononly[cols_only]
df_specificmbon = dfoneresponder[(dfoneresponder['responder']=="Chrimson2") & (dfoneresponder['MBON'].isin(specificmbons))].sort_values(by = "bspeed_deltag", ascending=True).reset_index(drop=True)
df_specificmbon.columns = df_specificmbon.columns.str.replace('_deltag', '')

df_singleresponder_lobeloc = addlobes(df_specificmbon)
tag = "Chrimson2_specific"

In [ ]:
plot_heatmap(df_singleresponder_lobeloc, responderrename, colormap, tag)

In [ ]:
plot_clustermap(df_singleresponder_lobeloc, responderrename, colormap, tag)

## 5. Differenced

In [ ]:
responderrename = "abs difference, CsChrimson vs GtACR1"
colormap = 'Greens'

counts = Counter(mbononly['MBON'].to_list())
matchingset = [value for value, count in counts.items() if count > 1]
dfreg_resp = mbononly[mbononly['MBON'].isin(matchingset)][cols_only].sort_values(by = ["MBON", "responder"]).reset_index(drop=True)
dfreg_resp.columns = dfreg_resp.columns.str.replace('_deltag', '')

effectsizediff = pd.DataFrame()
for n in matchingset:
    newt = dfreg_resp[dfreg_resp['MBON'] == n]
    MBONlabel = pd.DataFrame()
    MBONlabel['MBON'] = [n]
    newtdiff = newt.drop(['MBON', 'responder', 'genotypeandresponder'], axis = 1).diff().abs().iloc[1:,:].reset_index(drop=True)
    effectsizediff = pd.concat([effectsizediff, pd.concat([MBONlabel, newtdiff], axis = 1)]).reset_index(drop=True)

effectsizediff['responder'] = 'diff'
effectsizediff['genotypeandresponder'] = effectsizediff['MBON']

df_singleresponder_lobeloc = addlobes(effectsizediff)
tag = "absdiff"

In [ ]:
plot_heatmap(df_singleresponder_lobeloc, responderrename, colormap, tag, vmin=0, vmax=1.5)

In [ ]:
plot_clustermap(df_singleresponder_lobeloc, responderrename, colormap, tag, vmin=0, vmax=1.5)